# x6_biomas — Composição e Predominância de Biomas Municipais (Amazônia, Cerrado, Pantanal)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** participação de Amazônia, Cerrado e Pantanal, proporção 0–1, municipal, estática  
**Conceito e Mecanismo Econômico:** Fração territorial de cada bioma no município. Define o regime institucional de Reserva Legal (80% na Amazônia, 35% no Cerrado, 20% no Pantanal). Variável estática.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h2__biome_share__ibge2019_malha2024__candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar frações por bioma e classificações de predominância de 70%, 80% e 90% como alternativas.
> - Avaliar municípios mistos e suporte amostral; nenhum limiar está escolhido.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`biome_amazonia_share_i`**: Fração do município na Amazônia (0 a 1).
- **`biome_cerrado_share_i`**: Fração no Cerrado (0 a 1).
- **`biome_pantanal_share_i`**: Fração no Pantanal (0 a 1).
- **`biome_class_70`**: Bioma predominante corte 70%.
- **`biome_class_80`**: Bioma predominante corte 80%.
- **`biome_class_90`**: Bioma predominante corte 90%.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h2__biome_share__ibge2019_malha2024__candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x6_biomas`:
- **Atenção:** Explorar frações por bioma e classificações de predominância de 70%, 80% e 90% como alternativas.
- **Atenção:** Avaliar municípios mistos e suporte amostral; nenhum limiar está escolhido.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
# O arquivo candidato da malha 2024 contém 142 municípios (inclui Boa Esperança do Norte - 5101837).
# Reincorporamos 5101837 em Nova Ubiratã (5106240) e recalculamos as frações médias ponderadas.
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# Aglutinar Nova Ubiratã pela média das frações territoriais
df = df.groupby('codigo_ibge', as_index=False).agg({
    'biome_amazonia_share_i': 'mean',
    'biome_cerrado_share_i': 'mean',
    'biome_pantanal_share_i': 'mean'
})

# Reclassificar limiares 70%, 80% e 90%
for lim in [70, 80, 90]:
    cutoff = lim / 100.0
    df[f'biome_class_{lim}'] = 'misto'
    df.loc[df['biome_amazonia_share_i'] >= cutoff, f'biome_class_{lim}'] = 'amazonia'
    df.loc[df['biome_cerrado_share_i'] >= cutoff, f'biome_class_{lim}'] = 'cerrado'
    df.loc[df['biome_pantanal_share_i'] >= cutoff, f'biome_class_{lim}'] = 'pantanal'

print(f"Total de municípios de MT após harmonização: {df['codigo_ibge'].nunique()} (Meta: exatamente 141)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da fração de cada bioma nos 141 municípios de MT:")
print(df[['biome_amazonia_share_i', 'biome_cerrado_share_i', 'biome_pantanal_share_i']].describe())

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['biome_amazonia_share_i'], bins=20, ax=axes[0], color='green')
axes[0].set_title("Fração Amazônia")
sns.histplot(df['biome_cerrado_share_i'], bins=20, ax=axes[1], color='orange')
axes[1].set_title("Fração Cerrado")
sns.histplot(df['biome_pantanal_share_i'], bins=20, ax=axes[2], color='blue')
axes[2].set_title("Fração Pantanal")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
print("Classificação segundo limiar de 70%:")
print(df['biome_class_70'].value_counts())
print("\nClassificação segundo limiar de 80%:")
print(df['biome_class_80'].value_counts())

## Fase 6: Feature Engineering — Composição de Biomas e Choques Regulatórios

### 1. Step 1: Proporção Municipal por Bioma (IBGE)
- Proporção da área territorial de cada município localizada em cada um dos 3 biomas de MT:
  $ShareAmazonia_i$, $ShareCerrado_i$ e $SharePantanal_i$.
- Determinação do bioma predominante ($> 50\%$ da área municipal).

### 2. Step 2: Interação Institucional Pós-2008 (Código Florestal e Fiscalização)
- A legislação ambiental brasileira é assimétrica por bioma: a Reserva Legal obrigatória é de **80% na Amazônia** contra **35% no Cerrado**.
- Em 2008, o Decreto Federal 6.514 e a Resolução Bacen 3.545 criaram restrições severas de crédito rural e operações de fiscalização focadas na Amazônia.
- **Termo de Interação de Política:**
  $$InteracaoPolitica_{i, t} = ShareAmazonia_i \times Post2008_t$$
  Como o bioma é fixo no tempo (absorvido pelo efeito fixo municipal), essa interação permite identificar o impacto diferencial das políticas ambientais sobre a conversão!


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Dummies de bioma predominante
df_bioma = df.copy()

# 2. Interação Institucional com o Marco Regulatório Pós-2008
# Em painel com efeito fixo municipal (alpha_i), a fração estática do bioma é absorvida.
# A interação com o tempo (ano >= 2008) identifica o efeito do aperto regulatório na Amazônia.
def expandir_painel_bioma(df_bio, anos=range(2001, 2025)):
    registros = []
    for ano in anos:
        temp = df_bio.copy()
        temp['ano'] = ano
        # Post-2008: Decreto 6.514/2008, Lista de Municípios Prioritários, Resolução Bacen 3.545
        temp['post_2008'] = int(ano >= 2008)
        temp['interacao_amazonia_post2008'] = temp['biome_amazonia_share_i'] * temp['post_2008']
        registros.append(temp)
    return pd.concat(registros, ignore_index=True)

painel_bioma = expandir_painel_bioma(df_bioma)

print(f"Painel expandido de Biomas x Política: {painel_bioma.shape}")
print(painel_bioma[['codigo_ibge', 'ano', 'biome_amazonia_share_i', 'post_2008', 'interacao_amazonia_post2008']].head())